In [94]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [95]:
df = pd.read_csv("/content/lasso_regression_practice.csv")

In [96]:
df.head(2)

,area_sqft,bedrooms,bathrooms,age_years,distance_to_city_km,balcony_count,parking_spaces,floor_number,garden_area_sqft,noise_feature_1,noise_feature_2,noise_feature_3,price_lakh
0,1846,3,4,18,21.5,0,1,11,842,-0.528,-1.942,0.659,151.34
1,1622,5,2,15,15.0,0,1,15,82,-0.139,1.419,0.711,123.85


In [97]:
X = df.iloc[:,:-1]
y = df['price_lakh']

In [98]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [99]:
scaler = StandardScaler()
X_train_scaler = scaler.fit_transform(X_train)
X_test_scaler = scaler.transform(X_test)

In [110]:
SLR = LinearRegression()
SLR.fit(X_train , y_train)
SLR_Predict = SLR.predict(X_test)
r2_score(y_test , SLR_Predict)
mean_squared_error(y_test , SLR_Predict)

114.06201008145939

In [121]:
ElasticNet_model = ElasticNet(l1_ratio=0.95)
ElasticNet_model.fit(X_train_scaler , y_train)
ElasticNet_predict = ElasticNet_model.predict(X_test_scaler)
r2_score(y_test , ElasticNet_predict)

0.950287462133663

## Lets Create a Own Lasso Class With GD

In [122]:
class MeraElasticNet:

    def __init__(self, Lr, epochs, alpha, l1_ratio):

        self.Lr = Lr
        self.epochs = epochs
        self.alpha = alpha
        self.l1_ratio = l1_ratio

        # Calculate L1 and L2 strength
        self.lam1 = alpha * l1_ratio
        self.lam2 = alpha * (1 - l1_ratio)

        self.m = None
        self.b = None

    def fit(self, X_train, y_train):

        self.b = 0
        self.m = np.ones(X_train.shape[1])

        for i in range(self.epochs):

            for j in range(X_train.shape[0]):

                idx = np.random.randint(0, X_train.shape[0])

                y_hat = np.dot(X_train[idx], self.m) + self.b

                slope_loss_b = -2 * (y_train.iloc[idx] - y_hat)

                self.b = self.b - self.Lr * slope_loss_b

                slope_loss_m = (
                    -2 * np.dot(
                        (y_train.iloc[idx] - y_hat),
                        X_train[idx]
                    )
                    + 2 * self.lam1 * np.sign(self.m)
                    + 2 * self.lam2 * self.m
                )

                self.m = self.m - self.Lr * slope_loss_m

    def predict(self, X_test):

        y_pred = np.dot(X_test, self.m) + self.b

        return y_pred

In [127]:
model = MeraElasticNet(
    Lr=0.001,
    epochs=1000,
    alpha=0.01,
    l1_ratio=0.95
)
model.fit(X_train_scaler , y_train)
y_pred = model.predict(X_test_scaler)
r2_score(y_test , y_pred)

0.9522960024742547